# LIFE on Google Colab — GossipCop++ (all experiment tracks, top_k=15)

Runs the full LIFE experiment suite on the paper's larger benchmark, **GossipCop++**: binary MF-vs-MR (LLaMA2-7B / GPT-2 / Qwen2.5-32B), 4-class HF/HR/MF/MR, combined-by-veracity, and human-only HF-vs-HR. Same pipeline as the PolitiFact++ notebook — **convert → key-sentence extraction → concatenate → features → train** — just pointed at GossipCop++ with `top_k=15` (the paper's k for this dataset).

**Before you start:**
1. Set the Colab runtime to **GPU** — an **A100** is needed for LLaMA2-7B / Qwen2.5-32B (Runtime → Change runtime type).
2. Upload the whole `LIFE` repo (including `dataset/data/`) to your Google Drive, e.g. `MyDrive/LIFE`.
3. Edit `PROJECT_DIR` in the path cell below if you put it somewhere else.
4. Step 3 uses the **ungated** `NousResearch/Llama-2-7b-hf` mirror by default — no HF token needed.

**Scale warning:** GossipCop++ is ~36× larger than PolitiFact++ (8,253 LLM-pair articles: MF=4,084 fake + MR=4,169 real). Step 1 (a BERT forward pass per sentence per article) and Step 3 (LLaMA / Qwen-32B reconstruction) take **hours** on an A100 — run tracks one at a time and keep the runtime alive. All GossipCop artifacts are written under `dataset/gossipcop/` so they never overwrite the PolitiFact++ notebook's outputs. **Do not run both notebooks at the same time** — they share the same classifier checkpoint filenames (`linear_en.pt`, `bce_en.pt`, …); whichever ran last owns the file on disk.

In [ ]:
# Confirm a GPU is attached
!nvidia-smi

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

# <-- change this if you uploaded the repo elsewhere
PROJECT_DIR = '/content/drive/MyDrive/LIFE'
os.chdir(PROJECT_DIR)

GOSSIPCOP_DIR = f'{PROJECT_DIR}/dataset/data/Fakenews-dataset-main/Fakenews-dataset-main/Dataset/GossipCop++'

# Paper's binary task: LLM pair only (MF=fake, MR=real), reconstructed with LLaMA2-7B.
OUTPUT_BIN     = f'{PROJECT_DIR}/dataset/gossipcop/output_bin'        # MF_fake.jsonl + MR_true.jsonl
KEY_SENT       = f'{PROJECT_DIR}/dataset/gossipcop/keySentence/important_sentences_top15.jsonl'
BERT_CKPT      = f'{PROJECT_DIR}/dataset/gossipcop/bert_bin.pt'       # fresh extractor for MF-vs-MR
FEATURES_LLAMA = f'{PROJECT_DIR}/dataset/gossipcop/features_llama'
TRAIN_PATH     = f'{PROJECT_DIR}/dataset/gossipcop/train_bin.jsonl'
TEST_PATH      = f'{PROJECT_DIR}/dataset/gossipcop/test_bin.jsonl'

print('cwd:', os.getcwd())
print('GossipCop++ found:', os.path.isdir(GOSSIPCOP_DIR))

In [ ]:
# Install dependencies.
# If the fastNLP import fails at the training step, pin a compatible version:
#   !pip install -q fastNLP==1.0.1
!pip install -q -r requirements.txt

In [ ]:
# NLTK sentence tokenizer data. 'punkt' gives english.pickle (used by train.py);
# 'punkt_tab' is required by newer nltk's sent_tokenize (used by step 1).
import nltk
nltk.download('punkt')
nltk.download('punkt_tab')

## HuggingFace login (optional)
Step 3 defaults to the **ungated** `NousResearch/Llama-2-7b-hf` mirror, so **no token is needed — you can skip this cell**. Only run it if you switch Step 3 to the official gated `meta-llama/Llama-2-7b-hf` (which also requires an approved access request).

In [ ]:
# LLaMA-2 is gated on HuggingFace. First accept the license at
# https://huggingface.co/meta-llama/Llama-2-7b-hf, then run this cell and paste an
# access token from https://huggingface.co/settings/tokens
# (or replace with: login(token="hf_xxx")).
from huggingface_hub import login
login()

## Step 0 — Convert GossipCop++ to the binary LLM-pair JSONL
`--subset llm` emits only `MF_fake.jsonl` (4,084, fake) and `MR_true.jsonl` (4,169, real) — the paper's binary task. HF/HR (human-written) are not used.

In [ ]:
!python dataset/0_convert.py --input_dir "{GOSSIPCOP_DIR}" --output_dir "{OUTPUT_BIN}" --subset llm

## Step 1 — Key-sentence extraction (top-15)
Trains a **fresh** BERT fake/real classifier on MF-vs-MR (saved to `BERT_CKPT`), then keeps the **top-15** most impactful sentences per article (paper's k for GossipCop++). This is the slowest step (a forward pass per sentence per article) — **hours** on GossipCop++'s ~8k articles even on an A100.

In [ ]:
!python dataset/1_keySentenceExtraction.py --data_dir "{OUTPUT_BIN}" --output_file "{KEY_SENT}" --top_k 15 --model_path "{BERT_CKPT}" --gpu 0

## Step 2 — Concatenate key sentences back into the data
Adds a `sentence` field to each record in `OUTPUT_BIN` by matching on `(id, label)`. **Overwrites the files in `OUTPUT_BIN` in place** — re-run Step 0 first if you need to reset them.

In [ ]:
!python dataset/2_concate.py --folder_path "{OUTPUT_BIN}" --important_sentences_file "{KEY_SENT}"

## Step 3 — Reconstruction probabilities with LLaMA2-7B
The paper's reconstruction model. A malicious prompt is prepended and LLaMA2-7B's per-token log-likelihoods over the key fragments form the "linguistic fingerprint" features. Loads in bfloat16 (~14 GB; needs the A100) and downloads ~13 GB on first run. Writes one feature JSONL per input file into `FEATURES_LLAMA`. On GossipCop++ (~8k articles) this is a multi-hour run.

In [ ]:
# meta-llama/Llama-2-7b-hf is gated (needs Meta approval). NousResearch/Llama-2-7b-hf is an
# ungated mirror of the SAME weights/tokenizer — no token needed. Swap back to the official
# repo if/when your access request is approved.
!python dataset/3_gen_features_local.py --input_dir "{OUTPUT_BIN}" --output_dir "{FEATURES_LLAMA}" --model NousResearch/Llama-2-7b-hf --scorer llama --dtype bfloat16 --gpu 0

## Step 4A — train the classifier (released head: BMES tags + CRF + majority vote)
Splits `FEATURES_LLAMA` into train/test (seed-0, deterministic) and trains the released Transformer classifier for **50 epochs** on the binary MF-vs-MR task. This head diverges from the paper — it tags tokens with B/M/E/S labels, CRF-decodes them, and recovers the article label by majority vote — so it is the **A-side** of the head A/B. Step 4B below is the paper-faithful head. Paper target for GossipCop++: **Acc 0.937 / F1 0.924**.

In [ ]:
!python LIFE_train/train.py \
  --split_dataset \
  --data_path "{FEATURES_LLAMA}" \
  --train_path "{TRAIN_PATH}" \
  --test_path "{TEST_PATH}" \
  --model Transformer \
  --num_train_epochs 50

## Step 4B — paper head (sigmoid + BCE, Eq 11–12)
Same CNN→Transformer trunk, but the head matches the paper: masked mean-pool → one sigmoid probability per article, trained with **binary cross-entropy** (fake=1, real=0) and evaluated directly at article level — no BMES tags, no CRF, no majority vote. Files: `LIFE_train/model_bce.py` + `LIFE_train/train_bce.py` (the originals are untouched and remain the A-side).

**The A/B is fair:** 4A and 4B consume the *same* `FEATURES_LLAMA` and the *same* seed-0 train/test split (`--split_dataset` here regenerates the identical split, so running 4A first is not required). GossipCop++'s test set is ~1.2k articles, so per-seed variance is far smaller than PolitiFact++'s ~46; a single seed is fairly stable, though you can still sweep `--seed 1..4` for a mean ± std. A-side PolitiFact++ references: 85.5/80.8 and 83.9/78.2; GossipCop++ paper target 93.7/92.4. Checkpoint: `bce_en.pt`.

In [ ]:
!python LIFE_train/train_bce.py \
  --split_dataset \
  --data_path "{FEATURES_LLAMA}" \
  --train_path "{TRAIN_PATH}" \
  --test_path "{TEST_PATH}" \
  --num_train_epochs 50 \
  --seed 0

## Multiclass experiment — 4-class HF / HR / MF / MR (LLaMA2-7B, released CRF/BMES head)

A separate, exploratory run that classifies all **four** GossipCop++ categories
(human-fake, human-true, gpt3.5-fake, gpt3.5-true) instead of the paper's binary MF-vs-MR.
It reuses the released CRF/BMES head via `LIFE_train/train_multi.py` (a copy of `train.py`
with `en_labels` set to the four classes; `model.py` / `dataloader.py` are imported unchanged).

This needs its own LLaMA2-7B features (the binary `FEATURES_LLAMA` only has MF/MR), so
Steps 0m–3m re-run the pipeline with `--subset all` into separate `*_multi` paths — nothing
above is overwritten. Reference: an earlier 4-class run with **gpt2** features scored ~51.7%;
this swaps in the LLaMA2-7B features.

In [ ]:
# --- 4-class (HF/HR/MF/MR) experiment paths (separate from the binary run above) ---
OUTPUT_MULTI     = f'{PROJECT_DIR}/dataset/gossipcop/output_multi'
KEY_SENT_MULTI   = f'{PROJECT_DIR}/dataset/gossipcop/keySentence/important_sentences_multi_top15.jsonl'
BERT_CKPT_MULTI  = f'{PROJECT_DIR}/dataset/gossipcop/bert_multi.pt'
FEATURES_MULTI   = f'{PROJECT_DIR}/dataset/gossipcop/features_llama_multi'
TRAIN_PATH_MULTI = f'{PROJECT_DIR}/dataset/gossipcop/train_multi.jsonl'
TEST_PATH_MULTI  = f'{PROJECT_DIR}/dataset/gossipcop/test_multi.jsonl'

### Step 0m — Convert all four categories
`--subset all` emits `HF_fake` / `MF_fake` / `HR_true` / `MR_true` JSONL (the full GossipCop++ 4-class set — many thousands of articles).

In [ ]:
!python dataset/0_convert.py --input_dir "{GOSSIPCOP_DIR}" --output_dir "{OUTPUT_MULTI}" --subset all

### Step 1m — Key-sentence extraction (top-15)
Trains a fresh binary BERT (fake = HF+MF, true = HR+MR) and keeps the top-15 sentences per
article. Slowest step; now over the full GossipCop++ 4-class set — many thousands of articles, hours on an A100.

In [ ]:
!python dataset/1_keySentenceExtraction.py --data_dir "{OUTPUT_MULTI}" --output_file "{KEY_SENT_MULTI}" --top_k 15 --model_path "{BERT_CKPT_MULTI}" --gpu 0

### Step 2m — Concatenate key sentences
Adds the `sentence` field to the records in `OUTPUT_MULTI` **in place** — re-run Step 0m to reset.

In [ ]:
!python dataset/2_concate.py --folder_path "{OUTPUT_MULTI}" --important_sentences_file "{KEY_SENT_MULTI}"

### Step 3m — LLaMA2-7B reconstruction features
Same as the binary Step 3 but over all four files → `FEATURES_MULTI`.

In [ ]:
!python dataset/3_gen_features_local.py --input_dir "{OUTPUT_MULTI}" --output_dir "{FEATURES_MULTI}" --model NousResearch/Llama-2-7b-hf --scorer llama --dtype bfloat16 --gpu 0

### Step 4m — Train the 4-class classifier
Released CRF/BMES head over 16 BMES tags (4 classes × B/M/E/S), recovered to a 4-class label
by sentence majority vote. Prints Accuracy / Macro-F1 / per-class precision-recall in the
class id order printed at startup. Writes `linear_multi_en.pt`.

In [ ]:
!python LIFE_train/train_multi.py --split_dataset --data_path "{FEATURES_MULTI}" --train_path "{TRAIN_PATH_MULTI}" --test_path "{TEST_PATH_MULTI}" --model Transformer --num_train_epochs 50

### Step 4c — Combined-by-veracity binary (fake = HF+MF, real = HR+MR)
A second side experiment that **pools the human dataset in**: the four categories are
collapsed to two veracity buckets by label suffix (`LIFE_train/train_combined.py`, a copy of
`train.py` with a `DataManager` subclass that maps `_fake`→fake / `_true`→true). Released
CRF/BMES head, sentence majority-vote eval. Per-class P/R prints in the class id order
(fake=0, true=1) printed at startup.

**Reuses `FEATURES_MULTI`** from the 4-class experiment (features are label-agnostic) — run
Steps 0m–3m first. `--split_dataset` regenerates the same seed-0 split as Step 4m, so the two
runs are directly comparable. Reference: a prior combined run scored ~85.2 Acc / 78.4 Macro-F1
(human-written fakes lack the LLM fingerprint, which drags fake recall down).

In [ ]:
!python LIFE_train/train_combined.py --split_dataset --data_path "{FEATURES_MULTI}" --train_path "{TRAIN_PATH_MULTI}" --test_path "{TEST_PATH_MULTI}" --model Transformer --num_train_epochs 50

## GPT-2 reconstruction model — binary MF-vs-MR (both heads)

Swaps the reconstruction LM from LLaMA2-7B to **GPT-2** on the paper's binary task, run through
both heads, to isolate the effect of the reconstruction model. No code changes:
`3_gen_features_local.py` scores with GPT-2 via `--scorer bbpe`, and `train.py` / `train_bce.py`
are already the binary released / BCE heads.

**Reuses the existing `OUTPUT_BIN`** (the key sentences from the LLaMA binary run's Step 2) — do
**not** re-run Steps 0–2, so the only difference vs the LLaMA run is the reconstruction LM here.
(If `OUTPUT_BIN` was reset since, re-run the binary Steps 0–2 first.)

LLaMA baselines: released head ~83.9–85.5 / ~78–80.8; BCE head 86.82 ± 2.157 / 84.48 ± 2.986
(seeds 0–20).

In [ ]:
# GPT-2 binary paths (separate from the LLaMA binary run's FEATURES_LLAMA / TRAIN_PATH / TEST_PATH)
FEATURES_GPT2_BIN = f'{PROJECT_DIR}/dataset/gossipcop/features_gpt2_bin'
TRAIN_PATH_GPT2   = f'{PROJECT_DIR}/dataset/gossipcop/train_gpt2_bin.jsonl'
TEST_PATH_GPT2    = f'{PROJECT_DIR}/dataset/gossipcop/test_gpt2_bin.jsonl'

### Step 3g — GPT-2 reconstruction features
Scores the same `OUTPUT_BIN` articles with **GPT-2** (`--scorer bbpe`). GPT-2 is small — fast,
ungated, fits any GPU. Writes `MF_fake.jsonl` + `MR_true.jsonl` into `FEATURES_GPT2_BIN`.

In [ ]:
!python dataset/3_gen_features_local.py --input_dir "{OUTPUT_BIN}" --output_dir "{FEATURES_GPT2_BIN}" --model gpt2 --scorer bbpe --gpu 0

### Step 4A-gpt2 — released CRF/BMES head on GPT-2 features

In [ ]:
!python LIFE_train/train.py --split_dataset --data_path "{FEATURES_GPT2_BIN}" --train_path "{TRAIN_PATH_GPT2}" --test_path "{TEST_PATH_GPT2}" --model Transformer --num_train_epochs 50

### Step 4B-gpt2 — paper BCE head on GPT-2 features
Same GPT-2 features and the same deterministic seed-0 split as 4A. To match the LLaMA BCE
headline, re-run with `--seed 1..20` and report mean±std (~46-article test set ≈ 2 pts/article).

In [ ]:
!python LIFE_train/train_bce.py --split_dataset --data_path "{FEATURES_GPT2_BIN}" --train_path "{TRAIN_PATH_GPT2}" --test_path "{TEST_PATH_GPT2}" --num_train_epochs 50 --seed 0

## Qwen2.5-32B reconstruction model — binary MF-vs-MR (4-bit, both heads)

Third reconstruction-model comparison: **Qwen2.5-32B** loaded in **4-bit (nf4)** to fit a
single 40GB A100. Qwen uses GPT-2-style byte-level BPE, so the existing `--scorer bbpe` path
handles it; `--load_in_4bit` (new flag in `3_gen_features_local.py`) enables quantization.
`bitsandbytes` + `accelerate` are already in `requirements.txt`.

**Reuses `OUTPUT_BIN`** (same key sentences as the LLaMA and GPT-2 runs) — only the Step-3
reconstruction LM changes. Baselines to compare: LLaMA-2-7B (released ~83.9-85.5 / ~78-80.8;
BCE 86.82 / 84.48 mean); GPT-2 (BCE 83.6 / 78.1).

Notes: the base model `Qwen/Qwen2.5-32B` (not -Instruct) is the right choice for a perplexity
fingerprint. First run downloads the full bf16 weights (~65GB) then quantizes to ~18-22GB in
VRAM — the download is the slow part; loading afterward is quick. 4-bit logits differ slightly
from full precision, so don't mix these features with a non-quantized run in one split.

In [ ]:
# Qwen2.5-32B binary paths (separate from the LLaMA / GPT-2 runs)
FEATURES_QWEN_BIN = f'{PROJECT_DIR}/dataset/gossipcop/features_qwen_bin'
TRAIN_PATH_QWEN   = f'{PROJECT_DIR}/dataset/gossipcop/train_qwen_bin.jsonl'
TEST_PATH_QWEN    = f'{PROJECT_DIR}/dataset/gossipcop/test_qwen_bin.jsonl'

### Step 3q — Qwen2.5-32B reconstruction features (4-bit)
Scores the same `OUTPUT_BIN` articles with Qwen2.5-32B in 4-bit. Do **not** re-run Steps 0-2,
so the key sentences match the other reconstruction-model runs.

In [ ]:
!python dataset/3_gen_features_local.py --input_dir "{OUTPUT_BIN}" --output_dir "{FEATURES_QWEN_BIN}" --model Qwen/Qwen2.5-32B --scorer bbpe --load_in_4bit --gpu 0

### Step 4A-qwen — released CRF/BMES head on Qwen features

In [ ]:
!python LIFE_train/train.py --split_dataset --data_path "{FEATURES_QWEN_BIN}" --train_path "{TRAIN_PATH_QWEN}" --test_path "{TEST_PATH_QWEN}" --model Transformer --num_train_epochs 50

### Step 4B-qwen — paper BCE head on Qwen features
Same Qwen features + deterministic seed-0 split as 4A. Sweep `--seed 1..20` for mean+-std.

In [ ]:
!python LIFE_train/train_bce.py --split_dataset --data_path "{FEATURES_QWEN_BIN}" --train_path "{TRAIN_PATH_QWEN}" --test_path "{TEST_PATH_QWEN}" --num_train_epochs 50 --seed 0

## Human-only reconstruction — binary HF-vs-HR (LLaMA2-7B, both heads)

A **negative-control** experiment: train/test on the human dataset only — human_fake (HF) vs
human_true (HR). Both classes are human-written, so **neither** carries the prompt-induced LLM
fingerprint LIFE keys on; near-chance results are the expected, informative outcome (evidence
that LIFE detects LLM-*generation*, not fakeness per se). Reconstruction model: **LLaMA2-7B**.

Released head uses `LIFE_train/train_human.py` (a copy of `train.py` with `en_labels` = the two
human classes). The BCE head reuses `LIFE_train/train_bce.py` unchanged — its label is derived
from the `_fake`/`_true` suffix, so HF→fake, HR→real automatically.

**Reuses the 4-class `FEATURES_MULTI`** LLaMA features (which already contain `HF_fake.jsonl` +
`HR_true.jsonl`) — no re-scoring; this is a clean subset of the same features. Requires the
4-class Step 3m to have run; if `FEATURES_MULTI` is absent, run Steps 0m–3m first. Compare
against: MF-vs-MR (LLaMA) released ~83.9–85.5 / ~78–80.8, BCE 86.82 / 84.48.

In [ ]:
# Human-only binary paths (own namespace; nothing above is touched)
FEATURES_HUMAN   = f'{PROJECT_DIR}/dataset/gossipcop/features_llama_human'
TRAIN_PATH_HUMAN = f'{PROJECT_DIR}/dataset/gossipcop/train_human.jsonl'
TEST_PATH_HUMAN  = f'{PROJECT_DIR}/dataset/gossipcop/test_human.jsonl'

### Step 3h — Build the human-only features dir
Copies just the two human feature files out of `FEATURES_MULTI` into `FEATURES_HUMAN`
(no re-scoring — reuses the 4-class LLaMA features). `split_dataset` loads *every* `.jsonl` in
the folder, so `FEATURES_HUMAN` must contain **only** HF + HR.

In [ ]:
!mkdir -p "{FEATURES_HUMAN}" && cp "{FEATURES_MULTI}/HF_fake.jsonl" "{FEATURES_MULTI}/HR_true.jsonl" "{FEATURES_HUMAN}/"
!ls -la "{FEATURES_HUMAN}"

### Step 4A-human — released CRF/BMES head on human-only features
8 BMES tags (2 classes × B/M/E/S), sentence majority-vote eval. Per-class P/R prints in class id
order (human_fake=0, human_true=1). Writes `linear_human_en.pt`.

In [ ]:
!python LIFE_train/train_human.py --split_dataset --data_path "{FEATURES_HUMAN}" --train_path "{TRAIN_PATH_HUMAN}" --test_path "{TEST_PATH_HUMAN}" --model Transformer --num_train_epochs 50

### Step 4B-human — paper BCE head on human-only features
Reuses `train_bce.py` unchanged (HF→fake=1, HR→real=0 by suffix). Same human features and the
same deterministic seed-0 split as 4A. To match the BCE headline format, re-run `--seed 1..20`
and report mean±std.

In [ ]:
!python LIFE_train/train_bce.py --split_dataset --data_path "{FEATURES_HUMAN}" --train_path "{TRAIN_PATH_HUMAN}" --test_path "{TEST_PATH_HUMAN}" --num_train_epochs 50 --seed 0

## Notes / troubleshooting
- **HF gating**: Step 3 defaults to the ungated `NousResearch/Llama-2-7b-hf` mirror (no token). If you switch to the official `meta-llama` repo and hit a 403 "gated repo", your access request hasn't been approved yet.
- **fastNLP**: if Step 4 errors on `from fastNLP.modules.torch import ...`, run `!pip install -q fastNLP==1.0.1` and restart the runtime.
- **Checkpoints**: `BERT_CKPT` (step 1) and `linear_en.pt` (step 4) are written under `PROJECT_DIR` on Drive, so they survive disconnects.
- **NaN features**: if Step 3 prints NaN/inf, switch Step 3 to `--dtype float32` (fits the 40 GB A100).
- **Re-runs**: Step 2 mutates `OUTPUT_BIN` in place; always re-run Step 0 before re-running Steps 1–3 from scratch.
- **GossipCop++ scale**: ~36× PolitiFact++ (8,253 LLM-pair articles). Steps 1 and 3 take hours on an A100; run tracks sequentially and keep the runtime alive.
- **Shared checkpoints**: the classifier checkpoints (`linear_en.pt`, `bce_en.pt`, `linear_multi_en.pt`, `linear_human_en.pt`, …) use the same filenames as the PolitiFact++ notebook. They are transient (written then read within one run), but do **not** run both notebooks concurrently.